# Simulating pedigrees for kinship inference benchmarking

**Tristan Dennis, 2026**

We need to start simulating some pedigrees to benchmark our kinship estimator. This notebook explores simulating pedigrees with some digressions on ancestry and relatedness.

We are building a software package that estimates how related two individuals are from their DNA sequence data (genotypes). To test how well it works, we need:

1. **Simulated pedigrees** - family trees with known relationships (full siblings, half siblings, cousins, etc.)
2. **Ground-truth identity by descent (IBD)** - the exact segments of DNA shared between individuals, computed directly from the simulated family tree
3. **Synthetic genotype data** — realistic DNA sequences we can feed into our estimator as if they were real data

By comparing what our package *estimates* against what we *know to be true* from the simulation, we can measure its accuracy across different relationship types, species, and demographic histories. 

On a slightly more esoteric level that is good to know is that we can explore the impact of parameters like genome size, recombination rate and demography on expected IBD structure - which is helpful to know.

## To Dos
- Support for more complex demographies
- Make it easier to specify more complicated pedigrees (e.g. relatedness in specific inbreeding scenarios)
- Plug in familial to compare kinship coefficients est from IBD with those from familial
- Tidy up the pedigree specification / labelling (dict of pedigree structures. We could actually have this as an imported module or library.
- Maybe one thing we could do is we could include functionality to specify sim demographies and params so that familial can show people what their expected range of kinship coefficients may be for their organism?
- For the package and the paper we want to sim:
    - Human and mosquito whole genomes.
    - For a variety of different pedigrees.
    - From various demographies (to test inbreeding either explicitly through the pedigree or through the Demography).

### Done
- ~~Supply population allele frequencies to the estimator.~~ The genotype runs now
  also return `af` — per-locus allele frequencies estimated from an unrelated
  **reference panel** of `N_PANEL` diploids drawn from the same population (not
  from the related focal individuals, which would bias the estimator). `N_PANEL`
  is tunable: large ≈ true population AF (estimator ceiling), small = realistic
  AF estimation error. Next step is estimator-side: `coeffs.calculate_jacquard_coeffs`
  currently recomputes AF from the probands internally — it needs to accept
  `rep['af']` instead.

In [2]:
#Import dependencies
import sys
from pathlib import Path

import msprime
import tskit
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
import pandas as pd
import allel

# Import pedigree catalog it so we can look pedigrees up by id rather
# than redefining the builders in this notebook.
import pedigrees

## Part 1: Simulating inheritance and generating genotypes

### How inheritance works

When a parent passes DNA to a child, the child doesn't get one continuous block from each parent. Instead, the chromosome is broken at random points (**recombination**) and the child gets alternating chunks from each of the parent's two copies. This means that two full siblings, who share the same two parents, don't end up with *identical* DNA — they each got a different random mosaic of their parents' chromosomes. 

This randomness means that **the amount of DNA shared between relatives varies** even for a fixed relationship type. Two pairs of full siblings might share 45% of their DNA, or 55% — the average is 50% but there is real variance. This variance shrinks as genome size increases (more chromosomes = more genome = more averaging out of the randomness). 

It's complicated, but the recombination rate varies across the genome, and between species. And probably within populations of species. And maybe even between individuals. Generating recombination maps is labour intensive, computationally difficult and fraught. Here we will simplify everything by using a constant recombination rate for everything, and handle maps later.

### The two-phase simulation

We use a two-phase approach because two separate biological processes need to be modelled:

**Phase 1 — Pedigree inheritance** (`fixed_pedigree` model):
Simulates the recombination and transmission events within the family tree. This tells us *which chunks of DNA each individual inherited from which ancestor*: the **IBD structure**. This is our ground truth.

**Phase 2 — Population history** (Hudson coalescent):
The founders of the pedigree came from some population with its own evolutionary history. This phase simulates that history, determining what genetic variants (mutations) the founders carry. This gives us **realistic genotype data** for familial to work on.

These two phases are **biologically independent**: the IBD between siblings depends only on which recombination breakpoints occurred, not on what the DNA actually contains. This has an important consequence for how we design our simulations (see Part 3) and makes things both easier and a pain in some respects. 

**TL/DR: The ground truth is computed by our a priori knowledge of the shared IBD segments between pairs of individuals. The genotype data depend on background population history and will give us our realistic data for familial to test.**

In [ ]:
def plot_pedigree(tables):
    """Draw a pedigree from a TableCollection. Blue = founder, green = sample."""
    n = tables.individuals.num_rows

    # IndividualTableRow has no .nodes or .time attrs in tskit 1.0.3 —
    # derive both from the nodes table instead
    node_flags      = tables.nodes.flags
    node_individual = tables.nodes.individual
    node_times      = tables.nodes.time

    is_sample = [
        bool(np.any(node_flags[node_individual == i] & tskit.NODE_IS_SAMPLE))
        for i in range(n)
    ]
    times = [float(node_times[node_individual == i][0]) for i in range(n)]

    gen_members = defaultdict(list)
    for i, t in enumerate(times):
        gen_members[t].append(i)

    pos = {}
    for t, members in gen_members.items():
        for j, ind in enumerate(members):
            pos[ind] = (j - (len(members) - 1) / 2, -t)

    G = nx.DiGraph()
    labels = {i: f"ind {i}" for i in range(n)}
    colors = ["#A9DFBF" if is_sample[i] else "#AED6F1" for i in range(n)]
    for i in range(n):
        G.add_node(i)
        for parent in tables.individuals[i].parents:
            if parent >= 0:
                G.add_edge(parent, i)

    n_gen = len(gen_members)
    fig, ax = plt.subplots(figsize=(max(4, n), 2 * n_gen))
    nx.draw(G, pos=pos, labels=labels, node_color=colors,
            node_size=2000, font_size=8, arrows=True, arrowsize=15, ax=ax)
    ax.set_title("Pedigree  (blue = founder,  green = sample)")
    plt.tight_layout()
    plt.show()


def sim_pedigree(tables, r=1.1482e-08, seed=1):
    """Phase 1 — pedigree inheritance (fast, ~1s). Defaults to human-like recomb rate.

    Simulates recombination and transmission within the family tree. The
    result encodes exactly which chromosomal segments each individual
    inherited from which ancestor — this is our ground-truth IBD structure.

    Shared by both run types below, so the pedigree phase is defined once.
    """
    return msprime.sim_ancestry(
        initial_state=tables,
        model="fixed_pedigree",
        recombination_rate=r,
        random_seed=seed,
    )


def simulate(tables, demography=None, Ne=10_000, mu=1.29e-08, r=1.1482e-08, seed=1):
    """Run the full two-phase simulation and return (ts_ped, ts_mut). Defaults to human-like params.

    Phase 1 (sim_pedigree) gives ts_ped for ground-truth IBD.
    Phase 2 — population history (slower, ~15s at Ne=10,000, scales with Ne):
        Extends the founder lineages backward through a coalescent model,
        determining what genetic variants the founders carry. The mutated
        result (ts_mut) is the genotype data for the kinship estimator.

    Parameters
    ----------
    tables     : TableCollection from a pedigree builder function
    demography : msprime.Demography for the founder population.
                 Defaults to a single constant-size population with Ne=Ne.
                 Swap in a stdpopsim model for species-specific demography.
    Ne         : effective population size (only used if demography is None).
                 The simulation slows down as Ne increases.
    mu, r      : per-base mutation and recombination rates
    seed       : random seed (seed, seed+1, seed+2 used for the three steps)
    """
    if demography is None:
        demography = msprime.Demography.isolated_model([Ne])  # single constant-Ne pop

    ts_ped = sim_pedigree(tables, r=r, seed=seed)
    ts_full = msprime.sim_ancestry(
        initial_state=ts_ped,
        model="hudson",
        demography=demography,
        recombination_rate=r,
        random_seed=seed + 1,
    )
    ts_mut = msprime.sim_mutations(ts_full, rate=mu, random_seed=seed + 2)
    return ts_ped, ts_mut


def ibd_kinship(ts_ped, focal_ids):
    """Compute ground-truth kinship from pedigree IBD, for the focal samples only.

    IBD (Identity by Descent) means two individuals inherited the same stretch
    of DNA from a common ancestor. The kinship coefficient is the probability
    that a randomly chosen allele from individual A and one from individual B
    are IBD — i.e., came from the same ancestral copy.

    We compute this by summing total IBD across all four cross-individual
    haplotype pairs and dividing by 4 × genome length.

    max_time is set to the founder generation so only within-pedigree ancestry
    is counted — we ignore deeper shared history from the population phase.

    focal_ids restricts the calculation to the pedigree individuals of
    interest, so reference-panel samples (which are unrelated by construction)
    are not dragged into an O(panel^2) pairwise IBD computation.

    Returns
    -------
    dict of {(ind_a, ind_b): kinship_coefficient}
        Expected values: full siblings ≈ 0.25, half siblings ≈ 0.125,
        first cousins ≈ 0.0625, parent-offspring = 0.25
    """
    L = ts_ped.sequence_length
    founder_time = max(ind.time for ind in ts_ped.individuals())
    focal_nodes = [n for i in focal_ids for n in ts_ped.individual(i).nodes]
    ibd = ts_ped.ibd_segments(within=focal_nodes, max_time=founder_time, store_pairs=True)

    totals = {}
    for (na, nb), segs in ibd.items():
        ia, ib = ts_ped.node(na).individual, ts_ped.node(nb).individual
        if ia != ib:
            pair = tuple(sorted([ia, ib]))
            totals[pair] = totals.get(pair, 0.0) + segs.total_span

    return {pair: span / (4 * L) for pair, span in totals.items()}


def _individual_columns(ts):
    """Map each individual id -> its column indices into ts.genotype_matrix().

    genotype_matrix() columns are ordered by ts.samples() (one column per
    sample *node*). A diploid individual owns two sample nodes, hence two
    columns. This lets us pull out a given individual's two haplotypes.
    """
    sample_nodes = ts.samples()
    node_to_col = {n: c for c, n in enumerate(sample_nodes)}
    ind_cols = defaultdict(list)
    for n in sample_nodes:
        ind_cols[ts.node(n).individual].append(node_to_col[n])
    return ind_cols


def extract_genotypes_and_af(ts_mut, focal_ids, panel_ids,
                             restrict_to_panel_sites=True):
    """Build the two inputs a kinship estimator needs: genotypes + allele freqs.

    The estimator (familial / coeffs.calculate_jacquard_coeffs) needs both the
    focal individuals' genotypes AND population allele frequencies. Crucially
    the AFs must come from a population sample, NOT from the related focal
    individuals — estimating allele frequencies off the very individuals whose
    relatedness you are inferring biases the estimator. So we compute AFs from
    the unrelated reference panel added by the pedigree builders (pedigrees.py).

    Parameters
    ----------
    ts_mut      : mutated tree sequence from simulate()
    focal_ids   : individual IDs of the pedigree samples to genotype
    panel_ids   : individual IDs of the reference panel (for AF estimation)
    restrict_to_panel_sites : if True, keep only sites that are segregating in
                  the panel. This mirrors real data (you only genotype known
                  variant sites) and avoids zero-frequency alleles that would
                  make the estimator's likelihood blow up.

    Returns
    -------
    geno  np.int8   (n_focal, n_sites, 2)  individual-major, matches the
                                           estimator's expected (I, L, P) input
    af    np.float64 (n_sites, A)          per-locus allele frequency over the
                                           panel; row sums to 1; A = #alleles
    pos   np.int64  (n_sites,)             genomic bp position
    ref   np.str_   (n_sites,)             ancestral allele
    alt   object    (n_sites,)             tuple of alt alleles (len>1 = multiallelic)
    alleles object  (n_sites,)             full allele tuple per site (var.alleles),
                                           ancestral first followed by the alts
    """
    if len(panel_ids) == 0:
        raise ValueError("A reference panel is required to estimate allele "
                         "frequencies — call with n_panel > 0.")

    G = ts_mut.genotype_matrix()                       # (n_sites, n_sample_nodes)
    ind_cols = _individual_columns(ts_mut)
    A = int(G.max()) + 1                               # allele codes are 0..A-1

    panel_cols = [c for i in panel_ids for c in ind_cols[i]]
    panelG = G[:, panel_cols]                          # (n_sites, 2 * n_panel)

    if restrict_to_panel_sites:
        keep = panelG.max(axis=1) != panelG.min(axis=1)
    else:
        keep = np.ones(G.shape[0], dtype=bool)
    idx = np.flatnonzero(keep)

    panelG = panelG[keep]
    counts = np.stack([(panelG == a).sum(axis=1) for a in range(A)], axis=1).astype(np.float64)
    totals = counts.sum(axis=1, keepdims=True)
    af = np.divide(counts, totals, out=np.zeros_like(counts), where=totals > 0)

    geno = np.stack(
        [G[np.ix_(idx, ind_cols[i])] for i in focal_ids], axis=0
    ).astype(np.int8)                                  # (n_focal, n_sites, 2)

    allele_tuples = [v.alleles for v in ts_mut.variants()]
    pos = ts_mut.tables.sites.position.astype(int)[keep]
    ref = np.array([allele_tuples[k][0] for k in idx])
    alt = np.array([allele_tuples[k][1:] for k in idx], dtype=object)
    alleles = np.array([allele_tuples[k] for k in idx], dtype=object)  # full var.alleles per kept site

    return geno, af, pos, ref, alt, alleles

## Part 2: Two types of simulation run

We want to simulate two things:
    - A population history, or Demography. 

Because the pedigree phase and the population history phase are independent, we can separate them into two distinct workflows depending on what question we are asking.

---

### Run type 1 — IBD distribution (fast, many replicates)

**Question:** For a given relationship type and genome size, what is the *distribution* of actual IBD sharing?

Even though full siblings are expected to share 50% of their DNA on average, the actual amount varies between sibling pairs due to random recombination. To characterise this variance we run many replicate pedigree simulations with different random seeds. Each replicate represents an independent draw of recombination events.

**Key point:** we do not need the population history phase (Phase 2) here. IBD is determined entirely by the pedigree and recombination. This makes each replicate very fast (~1 second).

**What varies:** pedigree type, genome size, recombination rate.  
**What does not matter:** demography, mutation rate.

---

### Run type 2 — Estimator benchmarking (slower, fewer replicates)

**Question:** Given genotype data, how accurately does our kinship estimator recover the true IBD?

Here we need both phases: Phase 1 gives us the ground-truth IBD, and Phase 2 + mutations gives us the synthetic genotype data to feed into the estimator (familial). We can then compare the estimator's output against the ground truth.

**Key point:** the same `ts_ped` output from Phase 1 can be passed through Phase 2 with *different demographic models* to test estimator performance under different population histories — without re-running the (already fast) pedigree phase.

**What varies:** pedigree type, demography, mutation rate, genome size.  
**Ground truth is fixed** by the Phase 1 result.

In [ ]:
def run_pedigree_reps(pedigree_fn, seq_length=1e8, r=1.1482e-08, n_reps=100, seed=1):
    """Run many fast pedigree-only replicates to estimate the IBD distribution.
        Defaults to human like params.

    Each replicate is an independent draw of recombination events for the same
    relationship type. Only Phase 1 (sim_pedigree) is run — no population
    history, no mutations, no reference panel (n_panel=0). This makes each rep
    take ~1 second.

    Use this to answer: 'What is the distribution of realised IBD sharing
    for full siblings (or half siblings, cousins, etc.) on a genome of this
    size with this recombination rate?'

    Parameters
    ----------
    pedigree_fn : one of full_siblings, half_siblings, first_cousins, etc.
    seq_length  : genome length in base pairs (default SEQ_LENGTH)
    r           : per-base recombination rate (default R)
    n_reps      : number of independent replicates (default N_REPS)
    seed        : base random seed (seed+i used for rep i)

    Returns
    -------
    pd.DataFrame with columns [rep, ind_a, ind_b, kinship]
        one row per replicate per sample pair
    """
    records = []
    for i in range(n_reps):
        tables, focal_ids, _ = pedigree_fn(seq_length, n_panel=0)
        ts_ped = sim_pedigree(tables, r=r, seed=seed + i)
        for (ind_a, ind_b), k in ibd_kinship(ts_ped, focal_ids).items():
            records.append({'rep': i, 'ind_a': ind_a, 'ind_b': ind_b, 'kinship': k})
    return pd.DataFrame(records)


def run_genotype_reps(pedigree_fn, seq_length=1e8, n_panel=100, demography=None,
                      Ne=10_000, r=1.1482e-08, mu=1.29e-08, n_reps=10, seed=1,
                      restrict_to_panel_sites=True):
    """Run full two-phase replicates to produce genotype data for estimator testing.
    Defaults to human like params.

    Each replicate runs both simulation phases (via simulate) and returns:
      - ground-truth kinship (from Phase 1 IBD)
      - the focal individuals' genotypes (from Phase 2 + mutations)
      - population allele frequencies estimated from an unrelated reference
        panel of n_panel diploids drawn from the same population

    The genotypes + allele frequencies are exactly what the kinship estimator
    needs. By comparing the estimator output against the ground-truth kinship
    we can measure accuracy.

    n_panel is the tunable reference-panel size:
      - large n_panel  -> allele frequencies approach the true population
                          values, testing the estimator's best-case ceiling
      - small n_panel  -> realistic AF estimation error is included, testing
                          real-world performance
    The panel runs through the same coalescent + mutation step as the focal
    samples, so it shares the same sites (essential for AFs to line up).

    Swapping the demography argument lets you test the same pedigree under
    different population histories (e.g. bottleneck, expansion, or a
    species-specific stdpopsim model) without changing the pedigree structure.

    Note that if you increase Ne, it will slow down very quickly — bear in mind
    for mosquitoes. A larger n_panel also adds samples to the coalescent and
    slows Phase 2 down somewhat.

    Parameters
    ----------
    pedigree_fn : one of full_siblings, half_siblings, first_cousins, etc.
    seq_length  : genome length in base pairs (default SEQ_LENGTH)
    n_panel     : reference-panel size for allele frequency estimation
    demography  : msprime.Demography for founder pop.
                  Defaults to a constant-size pop with Ne=Ne.
                  Pass a stdpopsim model.model here for species-specific history.
    Ne          : effective population size (ignored if demography is provided)
    r, mu       : per-base recombination and mutation rates (default R, MU)
    n_reps      : number of independent replicates
    seed        : base random seed
    restrict_to_panel_sites : keep only sites segregating in the panel (see
                  extract_genotypes_and_af)

    Returns
    -------
    list of dicts, one per replicate, each containing:
        'rep'     : replicate index
        'kinship' : dict of {(ind_a, ind_b): true_kinship}
        'geno'    : np.int8 (n_focal, n_sites, 2) focal genotypes
        'af'      : np.float64 (n_sites, A) panel allele frequencies
        'pos'     : np.int64 (n_sites,) positions
        'ref'     : ancestral alleles
        'alt'     : alt alleles
        'alleles' : full var.alleles tuple per site (ancestral first, then alts)
    """
    results = []
    for i in range(n_reps):
        tables, focal_ids, panel_ids = pedigree_fn(seq_length, n_panel=n_panel)
        ts_ped, ts_mut = simulate(tables, demography=demography, Ne=Ne,
                                  mu=mu, r=r, seed=seed + i)
        geno, af, pos, ref, alt, alleles = extract_genotypes_and_af(
            ts_mut, focal_ids, panel_ids,
            restrict_to_panel_sites=restrict_to_panel_sites)
        results.append({
            'rep':     i,
            'kinship': ibd_kinship(ts_ped, focal_ids),
            'geno':    geno,
            'af':      af,
            'pos':     pos,
            'ref':     ref,
            'alt':     alt,
            'alleles': alleles,
        })
    return results

## Part 3: The pedigree catalog

A **pedigree** is a family tree. It specifies who the parents of each individual are. Rather than defining the relationship types inline here, they live in a shared catalog — `pedigrees.py` in the top-level directory — which we imported as `pedigrees` at the top of the notebook. This keeps the set of supported relationships in one place so it can grow independently of this notebook (and be reused by the benchmark scripts). The design echoes stdpopsim's own catalog: look an entry up by id, read its metadata, get the thing that builds it.

Each catalog entry is a `Pedigree` with an `id`, a `description`, an `expected_kinship`, and a `builder`. A builder takes `(seq_length, n_panel=0)` and returns `(tables, focal_ids, panel_ids)`:

- `tables` : an msprime `TableCollection` ready for `sim_ancestry`
- `focal_ids` : individual IDs of the pedigree samples we care about
- `panel_ids` : individual IDs of the unrelated reference panel (for allele-frequency estimation); pass `n_panel=0` for the pedigree-only IBD runs that don't need genotypes

Look entries up with `pedigrees.get_pedigree(id)` and list what's available with `pedigrees.list_pedigrees()`. To add or edit a relationship type, edit `pedigrees.py` rather than this notebook.

Conventions match msprime: each individual is assigned a **generation time** (0 for the present-day samples, 1 for parents, 2 for grandparents, …), and `time` runs *backwards* — 0 is the present, larger numbers are further in the past. Individuals with no parents are **founders**: the unrelated individuals that started the family line.

In [ ]:
# The pedigree builders now live in pedigrees.py (top-level dir) — see Part 3.
# Inspect the catalog: id, expected kinship, and description for each entry.
for pid in pedigrees.list_pedigrees():
    p = pedigrees.get_pedigree(pid)
    print(f"  {p.id:18} kinship~{p.expected_kinship:<7} {p.description}")

## Part 4: Run simulations

Here we actually run the simulations. Modify the constants below to alter the sim parameters.

In [ ]:
# ---------------------------------------------------------------------------
# Simulation constants — the single source of truth for every run in this
# modify these constants if you want to reparamaterise a sim
# TO - DO make this better so it's easier to rerun sim permutations in benchmarking phase
# OR we can add an optional call to load specific configs from stdpopsim
# ---------------------------------------------------------------------------
SEQ_LENGTH = 150_000_000   # genome length in bp (100 Mb)
R          = 1.3e-08       # per-base recombination rate
MU         = 3.5e-09       # per-base mutation rate
NE         = 10_000        # founder-population effective size
N_REPS     = 100           # replicates for the IBD distribution
N_GENO_REPS = 3            # replicates for genotype simulation
N_PANEL    = 100           # reference-panel size for allele-frequency estimation
                           #   large  -> AF approaches the true population value (ceiling)
                           #   small  -> realistic AF estimation error included

# Pick a relationship type by id from the catalog (see pedigrees.list_pedigrees()).
PEDIGREE_ID = 'full_sib'
ped = pedigrees.get_pedigree(PEDIGREE_ID)
focal_ped = ped.builder            # the (seq_length, n_panel) -> (tables, focal_ids, panel_ids) builder
ped_lab = ped.id

In [ ]:
# All parameters come from the constants defined at the top of the notebook
# (SEQ_LENGTH, R, MU, NE, N_REPS, N_GENO_REPS, N_PANEL).

# Pick a pedigree and inspect it (n_panel=0 here, we're just plotting structure)
tables, focal_ids, _ = focal_ped(SEQ_LENGTH, n_panel=0)
plot_pedigree(tables)

# Run type 1: IBD distribution (fast)
# N_REPS replicates, pedigree phase only — no population history needed.
# Each rep takes ~1s. Swap focal_ped to compare relationship types.
ibd_df = run_pedigree_reps(focal_ped, seq_length=SEQ_LENGTH, r=R, n_reps=N_REPS, seed=1)
print(f"IBD kinship distribution ({ped_lab}, {SEQ_LENGTH/1e6:.0f} Mb):")
print(ibd_df['kinship'].describe().round(4))

# Run type 2: genotype data + allele frequencies for estimator testing (slower)
# Each rep returns ground-truth kinship + focal genotypes + panel allele freqs.
geno_reps = run_genotype_reps(focal_ped, seq_length=SEQ_LENGTH, n_panel=N_PANEL,
                              Ne=NE, r=R, mu=MU, n_reps=N_GENO_REPS, seed=1)
print(f"\nGenotype reps (n_panel={N_PANEL}):")
for rep in geno_reps:
    k = list(rep['kinship'].values())[0]
    print(f"  rep {rep['rep']}: true kinship={k:.4f}  "
          f"sites={len(rep['pos']):,}  geno shape={rep['geno'].shape}  "
          f"af shape={rep['af'].shape}")

In [ ]:
# Plot histogram of kinship
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ibd_df['kinship'], bins=20, edgecolor='white')
ax.axvline(ped.expected_kinship, color='red', linestyle='--', label=f'expected ({ped.expected_kinship})')
ax.axvline(ibd_df['kinship'].mean(), color='orange', linestyle='--', label=f"mean ({ibd_df['kinship'].mean():.3f})")
ax.set_xlabel('Kinship coefficient')
ax.set_ylabel('Count')
ax.set_title(f'{ped_lab} IBD kinship - {N_REPS} reps, 100 Mb')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Jacob this should be your jumping off point for familial.
#
# Each rep now carries everything the estimator needs:
#   rep['geno'] : (n_focal, n_sites, 2) int8   <- coeffs wants individual-major (I, L, P)
#   rep['af']   : (n_sites, A) float           <- population allele freqs from the panel
#   rep['pos'] / rep['ref'] / rep['alt'] / rep['alleles']  <- rep['alleles'] is the full
#                                                             var.alleles tuple per site
#
# coeffs.calculate_jacquard_coeffs currently estimates allele frequencies from
# the focal individuals themselves — which is biased, because they're related.
# Feed rep['af'] in instead (computed from the unrelated reference panel).
rep = geno_reps[1]
geno, af = rep['geno'], rep['af']
print("geno (I, L, 2):", geno.shape, "   af (L, A):", af.shape)
print("alleles[:3]:", rep['alleles'][:3])

# scikit-allel expects (n_variants, n_samples, ploidy), so transpose if needed:
allel.GenotypeArray(geno.transpose(1, 0, 2))

In [ ]:
af

TODO 
- use different demos
- Jacob - each replicate of the genotype and demo sim will output a genotypearray for a given chromosome. We can feed this to familial to benchmark it with lots of replicates. Compare each replicate to the disttribution of benchmarked IBD sharing and we can get an estimate of how well familial is performing vs actual ancestry.
